# VAE for EHR Code Sequences

This is one notebook in a series of three about VAEs:

1. [Synthetic image VAE](../cxr/vae_cxr_synthetic.ipynb)
2. [ChestXray14 image VAE](../cxr/vae_cxr_chestxray14.ipynb)
3. [EHR code-sequence VAE (this notebook)](vae_ehr_timeseries.ipynb)

An earlier version of this same notebook was named `timeseries_vae_modeling.ipynb`. Despite the filename, its inputs are categorical EHR event sequences (for example, diagnosis codes), not numeric measurements such as vital signs. This example uses small synthetic sequences by default, so it runs without MIMIC access.


## What is a VAE, and why use one for EHR codes?

A **variational autoencoder (VAE)** learns a probability distribution over a compact latent representation of each input. It samples a latent vector and decodes it to reconstruct the input. Training balances reconstruction loss against a KL-divergence penalty that keeps latent distributions near a simple prior, encouraging a smooth latent space that can also be sampled to generate new examples.

For categorical EHR sequences, the encoder summarizes the ordered code tokens and an autoregressive decoder predicts them back one at a time. Potential research uses include learning patient representations, unsupervised pretraining, and studying synthetic code sequences. This example handles categorical sequences; it does not model numeric measurements, elapsed time, or full visit structure. Generated codes are not validated patient records and should not be treated as clinical recommendations or realistic synthetic data without further evaluation.

In PyHealth, the `SequenceProcessor` turns codes into vocabulary indices, while the VAE can train through the usual dataset and `Trainer` interfaces. This makes the model a reusable research baseline; the small synthetic example checks the workflow, not clinical usefulness.

## Build a synthetic sequence dataset

`SequenceProcessor` maps clinical-code strings to token IDs and pads batches. The mortality label is present only to mirror a common EHR task; the VAE learns to reconstruct its input sequence.


In [ ]:
import torch
from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import VAE
from pyhealth.trainer import Trainer

codes = ["I10", "E11", "J18", "N17", "I50"]
samples = [
    {"patient_id": f"p{i}", "conditions": codes[i % 5 : i % 5 + 1] + codes[: i % 3], "mortality": i % 2}
    for i in range(20)
]
dataset = create_sample_dataset(
    samples=samples,
    input_schema={"conditions": "sequence"},
    output_schema={"mortality": "binary"},
    dataset_name="vae-ehr-sequences-demo",
)
loader = get_dataloader(dataset, batch_size=4, shuffle=True, seed=7)
batch = next(iter(loader))
print("Padded token IDs:", batch["conditions"].shape)
print("Vocabulary:", dataset.input_processors["conditions"].code_vocab)

## Train and check reconstruction

The encoder summarizes each variable-length sequence. The decoder predicts the next token at each position; padding positions are excluded from the reconstruction loss.


In [ ]:
model = VAE(dataset, feature_key="conditions", latent_dim=8, hidden_dim=8)
trainer = Trainer(
    model=model,
    device="cuda" if torch.cuda.is_available() else "cpu",
    metrics=[],
)
trainer.train(
    train_dataloader=loader,
    epochs=2,
    optimizer_params={"lr": 1e-3},
)
with torch.no_grad():
    result = model(**batch)
print("Token-logit shape:", result["reconstruction_logits"].shape)
print("Reconstruction loss:", float(result["loss"]))

## Generate fixed-length code sequences

Generation is autoregressive and returns token IDs. The notebook maps those IDs back to the fitted vocabulary for inspection. Generated codes are a software demonstration, not validated synthetic clinical data.


In [ ]:
model.eval()
with torch.no_grad():
    generated = model.generate(num_samples=3, sequence_length=5)

vocab = dataset.input_processors["conditions"].code_vocab
token_by_id = {index: token for token, index in vocab.items()}
decoded = [[token_by_id[int(token)] for token in row] for row in generated.cpu()]
decoded

## Optional: use MIMIC-IV demo data

Set `PYHEALTH_MIMIC4_ROOT` to a local MIMIC-IV or MIMIC-IV demo directory to build the same sequence model from the built-in mortality task. Credentialed data stays on your machine.


In [ ]:
import os
from pathlib import Path

mimic_root = os.environ.get("PYHEALTH_MIMIC4_ROOT")
if mimic_root and Path(mimic_root).exists():
    from pyhealth.datasets import MIMIC4Dataset, get_dataloader
    from pyhealth.tasks import MortalityPredictionMIMIC4

    ehr = MIMIC4Dataset(
        ehr_root=mimic_root,
        ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions"],
        dev=True,
    )
    ehr_samples = ehr.set_task(MortalityPredictionMIMIC4())
    ehr_model = VAE(
        ehr_samples, feature_key="conditions", latent_dim=32, hidden_dim=32
    )
    ehr_loader = get_dataloader(ehr_samples, batch_size=16, shuffle=True)
    ehr_trainer = Trainer(model=ehr_model, metrics=[])
    ehr_trainer.train(train_dataloader=ehr_loader, epochs=1)
else:
    print("Set PYHEALTH_MIMIC4_ROOT to run the optional real-data example.")
